In [1]:
!pip install ultralytics

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.4/61.4 kB 4.4 MB/s eta 0:00:00
Mounted at /content/drive


**Step 1:** Exporting the model (fine-tuned PCB detection YOLO) in ONNX format.

In [2]:
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/pcb-detection-yolo/best.pt") # weights I kept from PCB fine-tune
model.export(format="onnx", dynamic=True, simplify=True)

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.120 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 122 layers, 2,379,321 parameters, 0 gradients, 5.3 GFLOPs

PyTorch: starting from '/content/drive/MyDrive/pcb-detection-yolo/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 450ms
Prepared 4 packag

'/content/drive/MyDrive/pcb-detection-yolo/best.onnx'

**Step 2:** Verifying parity (confirming the ONNX model still gives the same answers as the original)

In [3]:
pt = YOLO("/content/drive/MyDrive/pcb-detection-yolo/best.pt")
onnx = YOLO("/content/drive/MyDrive/pcb-detection-yolo/best.onnx")

img = "/content/drive/MyDrive/pcb-detection-yolo/pcb-inf-1.jpeg"
print("PT detections:", len(pt(img)[0].boxes))
print("ONNX detections:", len(onnx(img)[0].boxes))

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.

image 1/1 /content/drive/MyDrive/pcb-detection-yolo/pcb-inf-1.jpeg: 480x640 1 Button, 3 Capacitors, 4 ICs, 159.7ms
Speed: 20.6ms preprocess, 159.7ms inference, 5.0ms postprocess per image at shape (1, 3, 480, 640)
PT detections: 8
Loading /content/drive/MyDrive/pcb-detection-yolo/best.onnx for ONNX Runtime inference...
Using ONNX Runtime 1.28.0 with CPUExecutionProvider

image 1/1 /content/drive/MyDrive/pcb-detection-yolo/pcb-inf-1.jpeg: 480x640 1 Button, 3 Capacitors, 4 ICs, 113.0ms
Speed: 3.4ms preprocess, 113.0ms inference, 0.6ms postprocess per image at shape (1, 3, 480, 640)
ONNX detections: 8


In [6]:
!unzip -o '/content/drive/MyDrive/Roboflow Datasets/pcbv4yolo26.zip' -d '/content/pcb/'
!cat '/content/pcb/data.yaml'

!yolo val model="/content/drive/MyDrive/pcb-detection-yolo/best.onnx" data="/content/pcb/data.yaml"

Archive:  /content/drive/MyDrive/Roboflow Datasets/pcbv4yolo26.zip
  inflating: /content/pcb/README.dataset.txt  
  inflating: /content/pcb/README.roboflow.txt  
  inflating: /content/pcb/data.yaml  
 extracting: /content/pcb/test/images/ATTIOT_Bottom_jpg.rf.94cd89169043c7506cde7ced6de19680.jpg  
 extracting: /content/pcb/test/images/ATTIOT_Bottom_jpg.rf.a4f31e8c2d624c9e47b1e2f24ee88e69.jpg  
 extracting: /content/pcb/test/images/Arty_Bottom_jpg.rf.16f36245a0765afee4ea9dd43ae09f68.jpg  
 extracting: /content/pcb/test/images/Arty_Bottom_jpg.rf.b990509a5c76764e81f3e4e3a83b2711.jpg  
 extracting: /content/pcb/test/images/Arty_Top_jpg.rf.5d15b4645b32647b6439efa7fe4e3942.jpg  
 extracting: /content/pcb/test/images/Arty_Top_jpg.rf.7bc260a89099530b771500c983e1669e.jpg  
 extracting: /content/pcb/test/images/DuetWIFI_Top_png_jpg.rf.0b9a1e43b5f2dcefa35b5dee2de71274.jpg  
 extracting: /content/pcb/test/images/DuetWIFI_Top_png_jpg.rf.4167e818c7976f9ccb4729142142862a.jpg  
 extracting: /content/pc

The onnx run inference result is similar to the Pytorch result of 0.0877 (mAP50) and 0.0519 (mAP50-95).

**Step 3:** Downloading the ONNX model and test image locally for later use

In [10]:
from google.colab import files
files.download("/content/drive/MyDrive/pcb-detection-yolo/best.onnx")
files.download("/content/drive/MyDrive/pcb-detection-yolo/pcb-inf-1.jpeg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>